In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 23 — Retrieval Reranking Lab
## Compare No-Rerank vs LLM-Based Reranking

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Build a larger corpus where initial retrieval might bring back less relevant results
corpus = [
    Document(page_content="Python list comprehensions provide a concise way to create lists. "
        "The syntax is [expr for item in iterable if condition].", metadata={"id": 1}),
    Document(page_content="Python dictionaries store key-value pairs. Common methods: "
        "get(), keys(), values(), items(), update().", metadata={"id": 2}),
    Document(page_content="Python generators use yield to produce values lazily, saving "
        "memory for large datasets. Generator expressions use parentheses.", metadata={"id": 3}),
    Document(page_content="Python decorators are functions that modify other functions. "
        "Common use cases: logging, caching, auth, timing.", metadata={"id": 4}),
    Document(page_content="Python context managers handle setup and cleanup. Use the with "
        "statement. Implement via __enter__/__exit__ or @contextmanager.", metadata={"id": 5}),
    Document(page_content="Python async/await enables concurrent I/O operations. asyncio "
        "provides event loop, tasks, and coroutine management.", metadata={"id": 6}),
    Document(page_content="Python type hints improve code documentation: def greet(name: str) "
        "-> str. Use mypy for static type checking.", metadata={"id": 7}),
    Document(page_content="Python dataclasses reduce boilerplate for data containers. "
        "Use @dataclass decorator. Supports defaults, ordering, freezing.", metadata={"id": 8}),
]

vectorstore = Chroma.from_documents(corpus, embeddings, collection_name="rerank_lab")
print(f"Corpus indexed: {len(corpus)} documents")

## Step 2 — Initial Retrieval (No Reranking)

In [ ]:
def retrieve_no_rerank(query, k=5):
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [(doc, score) for doc, score in results]

query = "How do I efficiently process large amounts of data in Python?"

print(f"Query: {query}\n")
print("=== NO RERANKING (raw vector similarity) ===")
no_rerank_results = retrieve_no_rerank(query)
for i, (doc, score) in enumerate(no_rerank_results):
    print(f"  [{i+1}] score={score:.4f} id={doc.metadata['id']} — {doc.page_content[:60]}...")

## Step 3 — LLM-Based Reranker

In [ ]:
rerank_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a relevance judge. Given a query and a document,
rate the document's relevance on a scale of 0-10.
Return ONLY the number, nothing else.

Scoring guide:
0-2: Not relevant
3-5: Somewhat relevant
6-8: Relevant
9-10: Highly relevant"""),
    ("human", "Query: {query}\n\nDocument: {document}\n\nRelevance score (0-10):")
])
rerank_chain = rerank_prompt | llm | StrOutputParser()

def rerank_with_llm(query, initial_results, top_k=3):
    scored = []
    for doc, vector_score in initial_results:
        try:
            relevance = rerank_chain.invoke({"query": query, "document": doc.page_content})
            score = float(relevance.strip().split()[0])
        except (ValueError, IndexError):
            score = 0.0
        scored.append((doc, score, vector_score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

print("=== WITH LLM RERANKING ===")
reranked = rerank_with_llm(query, no_rerank_results)
for i, (doc, llm_score, vec_score) in enumerate(reranked):
    print(f"  [{i+1}] llm_score={llm_score:.0f} vec_score={vec_score:.4f} "
          f"id={doc.metadata['id']} — {doc.page_content[:60]}...")

## Step 4 — Side-by-Side Comparison

In [ ]:
test_queries = [
    "How do I efficiently process large amounts of data in Python?",
    "What's the best way to add caching to my functions?",
    "How do I handle concurrent network requests?",
    "What's a clean way to define data structures?",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")

    raw = retrieve_no_rerank(q, k=5)
    reranked = rerank_with_llm(q, raw, top_k=3)

    raw_order = [doc.metadata['id'] for doc, _ in raw[:3]]
    reranked_order = [doc.metadata['id'] for doc, _, _ in reranked]

    changed = raw_order != reranked_order
    print(f"  Raw top-3:     {raw_order}")
    print(f"  Reranked top-3: {reranked_order} {'← CHANGED' if changed else '(same)'}")

## Step 5 — End-to-End QA: No-Rerank vs Rerank

In [ ]:
from langchain.prompts import PromptTemplate

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Answer using the context.\nContext: {context}\nQuestion: {question}\nAnswer:"
)

def qa_no_rerank(query):
    docs = vectorstore.similarity_search(query, k=3)
    context = "\n".join([d.page_content for d in docs])
    chain = qa_prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": query})

def qa_with_rerank(query):
    raw = retrieve_no_rerank(query, k=5)
    reranked = rerank_with_llm(query, raw, top_k=3)
    context = "\n".join([d.page_content for d, _, _ in reranked])
    chain = qa_prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": query})

q = "How do I process large datasets efficiently in Python?"
print(f"Q: {q}\n")
print("Without reranking:")
print(qa_no_rerank(q))
print("\nWith LLM reranking:")
print(qa_with_rerank(q))

## What You Learned
- **Vector similarity** retrieval as a fast first pass
- **LLM reranking** for higher precision at the cost of latency
- **Relevance scoring** with structured LLM prompts
- **Side-by-side comparison** methodology